In [21]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
def create_dummy_compound_graph(num_nodes=25, num_edges=50):
    x = torch.randn(num_nodes, 44)
    edge_index = torch.randint(0, num_nodes, (2, num_edges))
    edge_attr = torch.randn(num_edges, 10)
    lap_pos_enc = torch.randn(num_nodes, 8)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, lap_pos_enc=lap_pos_enc)


def create_dummy_protein_graph(num_nodes=150, num_edges=300):
    x = torch.randn(num_nodes, 41)
    edge_index = torch.randint(0, num_nodes, (2, num_edges))
    edge_attr = torch.randn(num_edges, 5)
    lap_pos_enc = torch.randn(num_nodes, 8)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, lap_pos_enc=lap_pos_enc)


def create_dummy_protein_embedding(num_residues=150):
    return np.random.randn(num_residues, 320).astype(np.float32)


def create_dummy_dataset(num_samples=20):
    dataset = []
    for _ in range(num_samples):
        compound_nodes = np.random.randint(15, 40)
        compound_edges = compound_nodes * 2
        protein_nodes = np.random.randint(100, 200)
        protein_edges = protein_nodes * 2
        compound_graph = create_dummy_compound_graph(compound_nodes, compound_edges)
        protein_graph = create_dummy_protein_graph(protein_nodes, protein_edges)
        protein_embedding = create_dummy_protein_embedding(protein_nodes)
        label = np.random.uniform(4.0, 10.0)
        dataset.append({
            'compound_graph': compound_graph,
            'protein_graph': protein_graph,
            'protein_embedding': protein_embedding,
            'label': label
        })
    return dataset


class DTIDatasetPyG(Dataset):
    def __init__(self, data_list):
        self.data_list = data_list

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        item = self.data_list[idx]
        return item['compound_graph'], item['protein_graph'], item['protein_embedding'], item['label']

########### maybe be buggy follow original one 
print("hi")
def custom_collate_pyg(batch):  
    compound_graphs = [item[0] for item in batch]
    protein_graphs = [item[1] for item in batch]
    protein_embeddings = [item[2] for item in batch]
    labels = [item[3] for item in batch]

    compound_batch = Batch.from_data_list(compound_graphs)
    protein_batch = Batch.from_data_list(protein_graphs)

    max_protein_len = max(emb.shape[0] for emb in protein_embeddings)
    padded_embeddings = []
    for emb in protein_embeddings:
        if emb.shape[0] < max_protein_len:
            pad_size = max_protein_len - emb.shape[0]
            padded = np.pad(emb, ((0, pad_size), (0, 0)), mode='constant', constant_values=0)
            padded_embeddings.append(padded)
        else:
            padded_embeddings.append(emb)

    protein_embedding_batch = torch.FloatTensor(np.array(padded_embeddings))
    label_batch = torch.FloatTensor(labels)
    return compound_batch, protein_batch, protein_embedding_batch, label_batch


def demonstrate_shapes():
    dummy_data = create_dummy_dataset(num_samples=20)
    print(f"Created {len(dummy_data)} samples. Sample 0 embedding shape: {dummy_data[0]['protein_embedding'].shape}")

    dataset = DTIDatasetPyG(dummy_data)
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=custom_collate_pyg, drop_last=True)
    for batch_idx, data in enumerate(dataloader):
        compound_graph, protein_graph, protein_embedding, label = data
        print(f"Batch {batch_idx + 1}: compound_nodes={compound_graph.x.shape[0]}, protein_nodes={protein_graph.x.shape[0]}, embedding_shape={protein_embedding.shape}, labels={label.tolist()}")
        break


if __name__ == '__main__':
    # demonstrate_shapes()

    dummy_data = create_dummy_dataset(num_samples=8)
    print("\n================ DATASET LEVEL (First 4 Samples) ================\n")

    dataset = DTIDatasetPyG(dummy_data)

    for i in range(4):
        compound_graph, protein_graph, protein_embedding, label = dataset[i]

        print(f"--- Dataset Sample {i} ---")

        # Compound graph
        print("Compound Graph:")
        print(f"  Nodes           : {compound_graph.num_nodes}")
        print(f"  Edges           : {compound_graph.num_edges}")
        print(f"  Node feature    : {compound_graph.x.shape}")
        print(f"  Edge feature    : {compound_graph.edge_attr.shape}")
        print(f"  Lap PE          : {compound_graph.lap_pos_enc.shape}")

        # Protein graph
        print("\nProtein Graph:")
        print(f"  Nodes           : {protein_graph.num_nodes}")
        print(f"  Edges           : {protein_graph.num_edges}")
        print(f"  Node feature    : {protein_graph.x.shape}")
        print(f"  Edge feature    : {protein_graph.edge_attr.shape}")
        print(f"  Lap PE          : {protein_graph.lap_pos_enc.shape}")

        # Protein embedding
        print("\nProtein Embedding:")
        print(f"  Shape           : {protein_embedding.shape}")

        # Label
        print("\nLabel:")
        print(f"  Value           : {label:.4f}")

        print("\n")

    dataloader = DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=custom_collate_pyg)
    print("\n================ BATCH LEVEL (DataLoader Output) ================\n")

    for batch_idx, data in enumerate(dataloader):
        compound_graph, protein_graph, protein_embedding, label = data

        print(f"\n===== Batch {batch_idx + 1} =====")

        # Compound graph batch info
        print("Compound Graph Batch:")
        print(f"  Num graphs      : {compound_graph.num_graphs}")
        print(f"  Total nodes     : {compound_graph.num_nodes}")
        print(f"  Total edges     : {compound_graph.num_edges}")
        print(f"  Node feature    : {compound_graph.x.shape}")
        print(f"  Edge feature    : {compound_graph.edge_attr.shape}")
        print(f"  Lap PE          : {compound_graph.lap_pos_enc.shape}")

        # Protein graph batch info
        print("\nProtein Graph Batch:")
        print(f"  Num graphs      : {protein_graph.num_graphs}")
        print(f"  Total nodes     : {protein_graph.num_nodes}")
        print(f"  Total edges     : {protein_graph.num_edges}")
        print(f"  Node feature    : {protein_graph.x.shape}")
        print(f"  Edge feature    : {protein_graph.edge_attr.shape}")
        print(f"  Lap PE          : {protein_graph.lap_pos_enc.shape}")

        # Protein embedding batch info
        print("\nProtein Embedding Batch:")
        print(f"  Shape           : {protein_embedding.shape}")  
        print(f"  (batch, max_len, emb_dim)")

        # Labels
        print("\nLabels:")
        print(f"  Shape           : {label.shape}")
        print(f"  Values          : {label.tolist()}")

        # Device info
        print("\nDevices:")
        print(f"  Compound graph  : {compound_graph.x.device}")
        print(f"  Protein graph   : {protein_graph.x.device}")
        print(f"  Embedding       : {protein_embedding.device}")

        # Move to device
        compound_graph = compound_graph.to(device)
        protein_graph = protein_graph.to(device)
        protein_embedding = protein_embedding.to(device)
        label = label.to(device)

        print(f"\nMoved batch to device: {device}")

        break  # only inspect one batch




================ DATASET LEVEL (First 4 Samples) ================

--- Dataset Sample 0 ---
Compound Graph:
  Nodes           : 33
  Edges           : 66
  Node feature    : torch.Size([33, 44])
  Edge feature    : torch.Size([66, 10])
  Lap PE          : torch.Size([33, 8])

Protein Graph:
  Nodes           : 119
  Edges           : 238
  Node feature    : torch.Size([119, 41])
  Edge feature    : torch.Size([238, 5])
  Lap PE          : torch.Size([119, 8])

Protein Embedding:
  Shape           : (119, 320)

Label:
  Value           : 9.0487


--- Dataset Sample 1 ---
Compound Graph:
  Nodes           : 29
  Edges           : 58
  Node feature    : torch.Size([29, 44])
  Edge feature    : torch.Size([58, 10])
  Lap PE          : torch.Size([29, 8])

Protein Graph:
  Nodes           : 112
  Edges           : 224
  Node feature    : torch.Size([112, 41])
  Edge feature    : torch.Size([224, 5])
  Lap PE          : torch.Size([112, 8])

Protein Embedding:
  Shape           : (112, 320)